In [ ]:
import os, itertools, json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

IN_DIR = "/mnt/user-data/uploads"
OUT_DIR = "/mnt/user-data/outputs/stats_out"
FILES = dict(
    P1="P1_extra_all_results.csv", P2="P2_all-results.csv", P3="P3_all-results.csv",
    T1="P1_extra_All_tau.csv", T2="P2_all-layer-tau.csv", T3="P3_all-layer-tau.csv",
    LW="All_layerwise_results.csv", GS="ALL_gradsamples_results_1.csv", BS="bench_summary.csv")
ALPHA = 0.05
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)

NAMES = {"random_uniform": "Random", "frob_uniform": "Frobenius", "asicr_uniform": "A-SICR (op norm)",
         "asicr_full_uniform": "A-SICR-full", "grad_uniform": "Gradient", "asicr_global": "A-SICR (global)",
         "grad_global": "Gradient (global)", "asicr_la_kappa": "LA-A-SICR (kappa rule)", "asicr_la_L0": "LA-A-SICR (L0)",
         "asicr_inv_uniform": "A-SICR inverse", "frob_inv_uniform": "Frobenius inverse", "chain_inv_uniform": "A-SICR-full inverse"}
SET_LABEL = {("distilbert-base-uncased", "sst2"): "DistilBERT / SST-2 (acc)",
             ("distilbert-base-uncased", "cola"): "DistilBERT / CoLA (MCC)",
             ("distilbert-base-uncased", "mrpc"): "DistilBERT / MRPC (F1)",
             ("bert-base-uncased", "sst2"): "BERT-base / SST-2 (acc)"}
SETTINGS = list(SET_LABEL)
KS = [10, 8, 6, 4, 3, 2]
OKABE = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00", "#0072B2", "#CC79A7", "#999999"]
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 120, "savefig.dpi": 300, "legend.frameon": False})


def primary(df):
    col = np.where(df.task == "cola", df.val_mcc, np.where(df.task == "mrpc", df.val_f1, df.val_acc))
    return col * 100.0


def tci(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    n = len(x)
    if n == 0: return np.nan, np.nan, 0
    m = x.mean()
    if n < 2: return m, np.nan, n
    return m, stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n), n


def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan)
    idx = np.where(~np.isnan(p))[0]
    order = idx[np.argsort(p[idx])]
    m = len(order); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * p[i]); out[i] = min(1.0, run)
    return out


def signflip_p(d):
    d = np.asarray(d, float); d = d[~np.isnan(d)]
    n = len(d)
    if n < 2 or np.all(d == 0): return np.nan
    obs = abs(d.sum())
    s = np.array(list(itertools.product([-1, 1], repeat=n)))
    return float((np.abs((s * d).sum(1)) >= obs - 1e-12).mean())


def paired(a, b):
    """a, b: Series indexed by seed. Returns dict of paired statistics for a - b."""
    j = a.index.intersection(b.index)
    d = (a.loc[j] - b.loc[j]).values.astype(float)
    n = len(d)
    r = dict(n_seeds=n, mean_diff=d.mean() if n else np.nan, ci95=np.nan, p_t=np.nan, p_signflip=signflip_p(d),
             a_better=int((d > 0).sum()), ties=int((d == 0).sum()))
    if n >= 2:
        sd = d.std(ddof=1)
        r["ci95"] = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
        if sd > 0: r["p_t"] = float(stats.ttest_1samp(d, 0.0).pvalue)
    return r


def save(df, name, **kw):
    df.to_csv(f"{OUT_DIR}/tables/{name}.csv", index=False, **kw)
    return df


def figsave(fig, name):
    fig.savefig(f"{OUT_DIR}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT_DIR}/figures/{name}.pdf", bbox_inches="tight")
    plt.close(fig)


def rd(f): return pd.read_csv(f"{IN_DIR}/{f}")


P = pd.concat([rd(FILES["P1"]), rd(FILES["P2"]), rd(FILES["P3"])], ignore_index=True)
P["metric"] = primary(P)
dense = P[P.method == "dense"].groupby(["model", "task", "seed"]).metric.mean().rename("dense").reset_index()
S = (P[P.method != "dense"].groupby(["model", "task", "seed", "method", "k"]).metric.mean().reset_index())  # seed level
dense_mean = dense.groupby(["model", "task"]).dense.agg(["mean", "std", "count"])
print("Dense (primary metric x100):\n", dense_mean.round(2))
log = {}

rows = []
for (m, t), g in S.groupby(["model", "task"]):
    for (meth, k), gg in g.groupby(["method", "k"]):
        mean, hw, n = tci(gg.metric)
        rows.append(dict(model=m, task=t, method=meth, name=NAMES.get(meth, meth), k=k, frac_kept=k / 12, mean=mean, ci95_halfwidth=hw, n_seeds=n))
T1 = save(pd.DataFrame(rows), "T1_pruning_means_ci")

PAIRS = [("asicr_uniform", "frob_uniform"), ("asicr_uniform", "random_uniform"), ("asicr_full_uniform", "random_uniform"),
         ("asicr_full_uniform", "asicr_uniform"), ("grad_uniform", "asicr_full_uniform"), ("grad_uniform", "random_uniform"),
         ("asicr_inv_uniform", "asicr_uniform"), ("asicr_la_kappa", "asicr_uniform")]
rows = []
for a, b in PAIRS:
    fam = []
    for (m, t) in SETTINGS:
        g = S[(S.model == m) & (S.task == t)]
        for k in KS:
            sa = g[(g.method == a) & (g.k == k)].set_index("seed").metric
            sb = g[(g.method == b) & (g.k == k)].set_index("seed").metric
            if len(sa) and len(sb):
                fam.append(dict(model=m, task=t, a=a, b=b, k=k, **paired(sa, sb)))
    f = pd.DataFrame(fam)
    f["p_t_holm"] = holm(f.p_t.values)
    rows.append(f)
T2 = save(pd.concat(rows, ignore_index=True), "T2_paired_method_comparisons")
summ = []
for (a, b), g in T2.groupby(["a", "b"]):
    summ.append(dict(a=a, b=b, cells=len(g), a_sig_better_holm=int(((g.p_t_holm < ALPHA) & (g.mean_diff > 0)).sum()),
                     b_sig_better_holm=int(((g.p_t_holm < ALPHA) & (g.mean_diff < 0)).sum()),
                     a_sig_better_uncorrected=int(((g.p_t < ALPHA) & (g.mean_diff > 0)).sum()),
                     b_sig_better_uncorrected=int(((g.p_t < ALPHA) & (g.mean_diff < 0)).sum()),
                     mean_of_mean_diff=g.mean_diff.mean(), cells_a_higher_mean=int((g.mean_diff > 0).sum())))
T2s = save(pd.DataFrame(summ), "T2b_paired_summary")

rows = []
for (m, t), g in S.groupby(["model", "task"]):
    d = dense[(dense.model == m) & (dense.task == t)].set_index("seed").dense
    for (meth, k), gg in g.groupby(["method", "k"]):
        r = paired(gg.set_index("seed").metric, d)
        rows.append(dict(model=m, task=t, method=meth, k=k, **r))
T3 = pd.DataFrame(rows)
T3["p_t_holm"] = holm(T3.p_t.values)
T3["ci_includes_zero"] = (T3.mean_diff.abs() <= T3.ci95)
save(T3, "T3_vs_dense")

rows = []
for (m, t), g in S.groupby(["model", "task"]):
    res = g.groupby(["method", "k"]).metric.transform(lambda x: x - x.mean())
    res = res[g.groupby(["method", "k"]).metric.transform("count") >= 3]
    sw = stats.shapiro(res) if len(res) >= 3 else (np.nan, np.nan)
    rows.append(dict(model=m, task=t, test="Shapiro-Wilk on pooled cell-centred residuals", n=len(res), W=sw[0], p=sw[1]))
for (a, b) in PAIRS:
    dd = []
    for (m, t) in SETTINGS:
        g = S[(S.model == m) & (S.task == t)]
        for k in KS:
            sa = g[(g.method == a) & (g.k == k)].set_index("seed").metric
            sb = g[(g.method == b) & (g.k == k)].set_index("seed").metric
            j = sa.index.intersection(sb.index)
            if len(j) >= 3:
                x = (sa.loc[j] - sb.loc[j]).values
                dd += list(x - x.mean())
    if len(dd) >= 3:
        sw = stats.shapiro(dd)
        rows.append(dict(model="all", task="all", test=f"Shapiro-Wilk on pooled centred paired differences: {a} - {b}", n=len(dd), W=sw[0], p=sw[1]))
T4 = save(pd.DataFrame(rows), "T4_normality")

TAU = pd.concat([rd(FILES["T1"]), rd(FILES["T2"]), rd(FILES["T3"])], ignore_index=True)
rows, tau_tests = [], []
for (m, t), g in TAU.groupby(["model", "task"]):
    pv = g.p_op_vs_ablation.values
    ph = holm(pv)
    nt = len(g)
    for l, gl in g.groupby("layer"):
        rows.append(dict(model=m, task=t, layer=l, mean_tau_op=gl.tau_op_vs_ablation.mean(), sd=gl.tau_op_vs_ablation.std(ddof=1),
                         seeds_negative=int((gl.tau_op_vs_ablation < 0).sum()), seeds=len(gl), seeds_p_lt_05=int((gl.p_op_vs_ablation < ALPHA).sum()),
                         mean_tau_frob=gl.tau_frob_vs_ablation.mean(), mean_tau_op_vs_frob=gl.tau_op_vs_frob.mean(), mean_kappa=gl.mean_kappa.mean()))
    lm = g.groupby("layer").tau_op_vs_ablation.mean()
    tau_tests.append(dict(model=m, task=t, n_tests=nt, mean_tau_all=g.tau_op_vs_ablation.mean(), tests_p_lt_05=int((pv < ALPHA).sum()),
                          expected_by_chance=ALPHA * nt, tests_p_lt_05_holm=int((ph < ALPHA).sum()),
                          wilcoxon_p_layer_means_vs_0=float(stats.wilcoxon(lm).pvalue) if len(lm) >= 5 else np.nan,
                          ttest_p_layer_means_vs_0=float(stats.ttest_1samp(lm, 0).pvalue), n_layers=len(lm)))
T5 = save(pd.DataFrame(rows), "T5_kendall_by_layer")
T5b = save(pd.DataFrame(tau_tests), "T5b_kendall_summary")
n_h = 12
sd_tau = np.sqrt(2 * (2 * n_h + 5) / (9 * n_h * (n_h - 1)))
log["kendall_critical_abs_tau_n12_alpha05_normal_approx"] = float(stats.norm.ppf(0.975) * sd_tau)
# sanity check of the claim "op-vs-frob tau reproduces the old Table 5"
log["old_table5_check_layer0_tau_op_vs_frob_distilbert_sst2"] = float(
    TAU[(TAU.model == "distilbert-base-uncased") & (TAU.task == "sst2") & (TAU.layer == 0)].tau_op_vs_frob.mean())

LW = rd(FILES["LW"])
LW = LW[LW.layer >= 0].copy()
LWs = LW.groupby(["model", "task", "seed", "layer", "k", "method"]).agg(drop=("drop", "mean"), kappa=("layer_mean_kappa", "mean")).reset_index()
LWs["drop"] *= 100
rows = []
for (m, t), g in LWs[LWs.k == 0].groupby(["model", "task"]):
    for l, gl in g.groupby("layer"):
        mean, hw, n = tci(gl["drop"])
        rows.append(dict(model=m, task=t, layer=l, drop_points=mean, ci95_halfwidth=hw, n_seeds=n, layer_mean_kappa=gl.kappa.mean()))
T6 = save(pd.DataFrame(rows), "T6_layer_sensitivity_k0")


def partial_spearman(x, y, z):
    rx, ry, rz = [stats.rankdata(v) for v in (x, y, z)]
    def res(a, b):
        A = np.c_[np.ones(len(b)), b]
        return a - A @ np.linalg.lstsq(A, a, rcond=None)[0]
    return float(stats.pearsonr(res(rx, rz), res(ry, rz))[0])


rows = []
for (m, t), g in T6.groupby(["model", "task"]):
    rho, p = stats.spearmanr(g.layer_mean_kappa, g.drop_points)
    rd_, pd_ = stats.spearmanr(g.layer, g.drop_points)
    rk, pk = stats.spearmanr(g.layer, g.layer_mean_kappa)
    rows.append(dict(model=m, task=t, n_layers=len(g), rho_kappa_vs_drop=rho, p=p, rho_depth_vs_drop=rd_, p_depth=pd_,
                     rho_kappa_vs_depth=rk, partial_rho_kappa_drop_given_depth=partial_spearman(g.layer_mean_kappa.values, g.drop_points.values, g.layer.values)))
T6b = save(pd.DataFrame(rows), "T6b_kappa_vs_sensitivity")

wl = LWs[(LWs.k > 0)].groupby(["model", "task", "seed", "k", "method"])["drop"].mean().reset_index()
rows = []
for (m, t) in SETTINGS:
    g = wl[(wl.model == m) & (wl.task == t)]
    for k in sorted(g.k.unique()):
        gk = g[g.k == k]
        sr = gk[gk.method == "random"].set_index("seed")["drop"]
        for meth in ["frob", "op", "chain", "grad"]:
            sm = gk[gk.method == meth].set_index("seed")["drop"]
            r = paired(sr, sm)       # positive = random loses MORE than meth = meth better
            mean, hw, n = tci(sm)
            rows.append(dict(model=m, task=t, k=k, method=meth, mean_drop=mean, ci95=hw, random_minus_method=r["mean_diff"],
                             ci95_diff=r["ci95"], p_t=r["p_t"], method_better_seeds=r["a_better"], n_seeds=r["n_seeds"]))
T6c = pd.DataFrame(rows)
T6c["p_t_holm"] = holm(T6c.p_t.values)
save(T6c, "T6c_within_layer_criteria_vs_random")

GS = rd(FILES["GS"])
GS["metric"] = GS.primary * 100
GSs = GS[GS.method != "dense"].groupby(["model", "task", "seed", "method", "n", "k"]).agg(metric=("metric", "mean"), tau=("tau_to_ref", "mean"),
                                                                                         overlap=("overlap_to_ref", "mean"), sec=("score_seconds", "mean")).reset_index()
rows = []
for (m, t, meth, n, k), g in GSs.groupby(["model", "task", "method", "n", "k"]):
    mean, hw, c = tci(g.metric)
    rows.append(dict(model=m, task=t, method=meth, n=n, k=k, mean=mean, ci95=hw, n_seeds=c, tau_to_ref=g.tau.mean(), overlap_to_ref=g.overlap.mean(), score_seconds=g.sec.mean()))
T7 = save(pd.DataFrame(rows), "T7_gradsamples_means")
ref_n = GS[(GS.method == "grad") & (GS.draw == -1)].groupby(["model", "task"]).n.first().to_dict()
rows = []
for (m, t) in SETTINGS:
    g = GSs[(GSs.model == m) & (GSs.task == t)]
    rn = ref_n[(m, t)]
    nl = sorted(n for n in g[g.method == "act"].n.unique())
    for k in sorted(g.k.unique()):
        gk = g[g.k == k]
        chain = gk[gk.method == "chain"].set_index("seed").metric
        for n in nl:
            ga = gk[(gk.method == "act") & (gk.n == n)].set_index("seed").metric
            gg = gk[(gk.method == "grad") & (gk.n == n)].set_index("seed").metric
            rows.append(dict(model=m, task=t, k=k, n=n, is_reference_n=(n == rn), comparison="act - grad", **paired(ga, gg)))
            rows.append(dict(model=m, task=t, k=k, n=n, is_reference_n=(n == rn), comparison="act - chain(data-free)", **paired(ga, chain)))
            rows.append(dict(model=m, task=t, k=k, n=n, is_reference_n=(n == rn), comparison="grad - chain(data-free)", **paired(gg, chain)))
T7b = pd.DataFrame(rows)
T7b["p_t_holm"] = T7b.groupby("comparison").p_t.transform(lambda s: pd.Series(holm(s.values), index=s.index))
save(T7b, "T7b_gradsamples_paired")

Pn = P[P.method != "dense"].copy()
rows = []
for (m, t), g in Pn.groupby(["model", "task"]):
    for proxy in ["C_op", "C_fro"]:
        pooled = stats.spearmanr(g[proxy], g.metric)[0]
        within = [stats.spearmanr(gk[proxy], gk.metric)[0] for k, gk in g.groupby("k") if len(gk) > 5]
        rows.append(dict(model=m, task=t, proxy=proxy, rho_pooled_over_budgets=pooled, mean_rho_within_budget=np.nanmean(within),
                         min_within=np.nanmin(within), max_within=np.nanmax(within), n_rows=len(g)))
T8 = save(pd.DataFrame(rows), "T8_proxy_vs_accuracy")
gap = Pn.assign(gap_loss=Pn.gap_loss).groupby(["model", "task", "method", "k"]).gap_loss.mean().reset_index()
save(gap[gap.method.isin(["random_uniform", "asicr_uniform", "grad_uniform", "frob_uniform", "asicr_full_uniform"])], "T8b_generalisation_gap_loss")

BS = rd(FILES["BS"])
T9 = save(BS[["model", "k", "heads_total", "params_m", "flops_attn_rel", "flops_total_rel", "gpu_lat_b1_ms_median", "gpu_lat_b1_speedup",
              "gpu_thr_b128_sps", "gpu_thr_b128_min", "gpu_thr_b128_max", "gpu_thr_b128_ratio", "cpu_lat_b1_ms_median", "cpu_lat_b1_speedup",
              "cpu_thr_b32_sps", "cpu_thr_b32_ratio", "gpu_weight_mb", "weight_mem_reduction_pct", "gpu_peak_mem_mb", "peak_mem_reduction_pct",
              "gpu_energy_j_per_1000", "energy_gross_rel", "co2_g_per_1M_inferences", "val_acc_fp16"]].sort_values(["model", "k"]), "T9_efficiency")

MAIN = ["random_uniform", "frob_uniform", "asicr_uniform", "asicr_full_uniform", "grad_uniform", "asicr_la_kappa"]
STY = {"random_uniform": ("#999999", "o", "-"), "frob_uniform": ("#E69F00", "s", "-"), "asicr_uniform": ("#D55E00", "^", "-"),
       "asicr_full_uniform": ("#009E73", "D", "-"), "grad_uniform": ("#0072B2", "v", "-"), "asicr_la_kappa": ("#CC79A7", "P", "--")}
# F1
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for ax, (m, t) in zip(axes.ravel(), SETTINGS):
    g = T1[(T1.model == m) & (T1.task == t)]
    dm = dense_mean.loc[(m, t), "mean"]
    ax.axhline(dm, color="k", ls=":", lw=1.2, label="Dense")
    for meth in MAIN:
        gg = g[g.method == meth].sort_values("frac_kept")
        c, mk, ls = STY[meth]
        ax.errorbar(gg.frac_kept * 100, gg["mean"], yerr=gg.ci95_halfwidth, color=c, marker=mk, ls=ls, ms=5, lw=1.4, capsize=2, label=NAMES[meth])
    ax.set_title(SET_LABEL[(m, t)]); ax.set_xlabel("Heads kept (%)"); ax.set_ylabel("Validation score x100")
    ax.invert_xaxis()
axes[0, 0].legend(fontsize=8, loc="lower left")
fig.suptitle("Accuracy vs. heads kept (mean and 95% t-interval over fine-tuning seeds)")
figsave(fig, "F1_accuracy_vs_heads_kept")

# F2 layer sensitivity
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for ax, (m, t) in zip(axes.ravel(), SETTINGS):
    g = T6[(T6.model == m) & (T6.task == t)].sort_values("layer")
    ax.bar(g.layer, g.drop_points, yerr=g.ci95_halfwidth, color="#0072B2", alpha=0.85, capsize=2)
    ax.set_xlabel("Layer whose attention block is removed"); ax.set_ylabel("Drop in score (points)"); ax.set_title(SET_LABEL[(m, t)])
    ax2 = ax.twinx(); ax2.plot(g.layer, g.layer_mean_kappa, "o--", color="#D55E00", ms=4); ax2.set_ylabel("Layer mean kappa", color="#D55E00"); ax2.grid(False)
    ax2.spines["right"].set_visible(True)
    r = T6b[(T6b.model == m) & (T6b.task == t)].iloc[0]
    ax.text(0.02, 0.95, f"Spearman(kappa, drop) = {r.rho_kappa_vs_drop:.2f}, p = {r.p:.2f} (n={int(r.n_layers)} layers)", transform=ax.transAxes, fontsize=8, va="top")
figsave(fig, "F2_layer_sensitivity")

fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
KFIG = 4
for ax, (m, t) in zip(axes.ravel(), SETTINGS):
    g = T7[(T7.model == m) & (T7.task == t) & (T7.k == KFIG)]
    for meth, c, mk, lab in [("grad", "#0072B2", "v", "Gradient (labelled)"), ("act", "#009E73", "D", "Activation (unlabelled)")]:
        gg = g[g.method == meth].sort_values("n")
        ax.errorbar(gg.n, gg["mean"], yerr=gg.ci95, color=c, marker=mk, ms=5, capsize=2, label=lab)
    for meth, c, ls, lab in [("chain", "#CC79A7", "--", "A-SICR-full (data-free)"), ("op", "#D55E00", "--", "A-SICR (data-free)"), ("random", "#999999", "--", "Random")]:
        v = g[(g.method == meth)]["mean"]
        if len(v): ax.axhline(v.iloc[0], color=c, ls=ls, lw=1.2, label=lab)
    ax.axhline(dense_mean.loc[(m, t), "mean"], color="k", ls=":", lw=1.2, label="Dense")
    ax.set_xscale("log"); ax.set_xlabel("Examples used for scoring (n)"); ax.set_ylabel("Validation score x100"); ax.set_title(SET_LABEL[(m, t)] + f", {KFIG} heads/layer kept")
axes[0, 0].legend(fontsize=7.5, loc="lower right")
figsave(fig, "F3_scoring_data_requirements")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for m, c in [("distilbert-base-uncased", "#0072B2"), ("bert-base-uncased", "#D55E00")]:
    g = T9[T9.model == m].sort_values("k")
    short = "DistilBERT" if m.startswith("distil") else "BERT-base"
    x = g.k / 12 * 100
    axes[0].plot(x, g.gpu_thr_b128_ratio, "o-", color=c, label=f"{short} GPU thr. (b128)")
    axes[0].plot(x, g.cpu_thr_b32_ratio, "s--", color=c, label=f"{short} CPU thr. (b32)")
    axes[1].plot(x, g.energy_gross_rel, "o-", color=c, label=f"{short} energy per inference")
    axes[1].plot(x, g.flops_total_rel, "s--", color=c, label=f"{short} FLOPs (analytic)")
    axes[2].plot(x, g.weight_mem_reduction_pct, "o-", color=c, label=f"{short} weights")
    axes[2].plot(x, g.peak_mem_reduction_pct, "s--", color=c, label=f"{short} peak inference")
for a, yl in zip(axes, ["Relative to dense (higher = faster)", "Relative to dense (lower = cheaper)", "GPU memory reduction (%)"]):
    a.set_xlabel("Heads kept (%)"); a.set_ylabel(yl); a.invert_xaxis(); a.legend(fontsize=7)
axes[0].axhline(1, color="k", lw=0.8, ls=":"); axes[1].axhline(1, color="k", lw=0.8, ls=":")
figsave(fig, "F4_efficiency")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)
for ax, (m, t) in zip(axes, [("distilbert-base-uncased", "sst2"), ("bert-base-uncased", "sst2")]):
    b = T9[T9.model == m].set_index("k").gpu_thr_b128_ratio
    for meth in ["random_uniform", "asicr_uniform", "asicr_full_uniform", "grad_uniform"]:
        gg = T1[(T1.model == m) & (T1.task == t) & (T1.method == meth)].sort_values("k")
        c, mk, ls = STY[meth]
        ax.plot([b[k] for k in gg.k], gg["mean"], marker=mk, color=c, ls=ls, label=NAMES[meth])
    ga = T7[(T7.model == m) & (T7.task == t) & (T7.method == "act") & (T7.n == ref_n[(m, t)])].sort_values("k")
    ax.plot([b[k] for k in ga.k], ga["mean"], marker="*", color="#000000", ms=8, ls="-", label="Activation (unlabelled)")
    ax.axhline(dense_mean.loc[(m, t), "mean"], color="k", ls=":", lw=1.2)
    ax.set_xlabel("GPU throughput relative to dense (batch 128)"); ax.set_ylabel("Validation accuracy x100"); ax.set_title(SET_LABEL[(m, t)])
axes[0].legend(fontsize=8, loc="lower left")
figsave(fig, "F5_accuracy_vs_speedup")

sh = T4[T4.test.str.startswith("Shapiro-Wilk on pooled cell")]
txt = f"""STATISTICAL METHODS (draft for the paper)
Replication unit. Each pruning configuration was evaluated on {int(S.groupby(['model','task']).seed.nunique().min())}-{int(S.groupby(['model','task']).seed.nunique().max())} independently seeded fine-tuning runs of the same model and task. Random pruning was repeated 5 times per fine-tuned model and averaged within the seed before any statistic was computed.
Intervals. All 95% confidence intervals are Student-t intervals over seeds (n = 3 for BERT-base, n = 5 otherwise).
Comparisons. Two criteria were compared with a paired t-test on per-seed differences. Because n is small we also report an exact sign-flip test (smallest attainable two-sided p = 0.0625 at n = 5). Within each family (one pair of criteria, all model-task settings and budgets), p-values were adjusted with the Holm step-down procedure; both uncorrected and corrected values are reported. alpha = 0.05 (two-sided).
Assumptions. Normality was assessed with Shapiro-Wilk on pooled, cell-centred residuals (the individual cells are too small for a meaningful per-cell test): p {', '.join(('< 0.001' if p < 0.001 else f'= {p:.3f}') for p in sh.p)} for the four settings, i.e. normality is rejected for the pooled residuals (partly because the variance differs strongly between cells, e.g. between a mild budget and a collapsed one). The t-intervals are therefore approximate; with n <= 5 per cell the per-cell checks have almost no power, so we also report the exact sign-flip test and describe small p-values as indicative.
Rank correlation. Kendall's tau between per-head operator norm and single-head ablation effect (12 heads per layer) is rank-based and makes no distributional assumption. A single layer-level test with n = 12 heads is significant at alpha = 0.05 (two-sided) when |tau| > about {log['kendall_critical_abs_tau_n12_alpha05_normal_approx']:.2f} (normal approximation); because the same checkpoint family yields near-identical head rankings across seeds, per-seed tests are not independent, so we report the number of nominally significant tests against the number expected by chance and a Holm-corrected count.
Layer-level correlations (kappa vs sensitivity) use Spearman's rho over layers (n = 6 or 12 layers); with so few layers they are descriptive.
"""
open(f"{OUT_DIR}/statistical_methods.txt", "w").write(txt)
json.dump(log, open(f"{OUT_DIR}/log.json", "w"), indent=2)

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("\n== Normality =="); print(T4.round(3).to_string(index=False))
print("\n== Paired summary (Holm within family) =="); print(T2s.round(2).to_string(index=False))
print("\n== Kendall summary =="); print(T5b.round(3).to_string(index=False)); print(log)
print("\n== kappa vs sensitivity =="); print(T6b.round(3).to_string(index=False))
print("\n== proxy vs accuracy =="); print(T8.round(2).to_string(index=False))
print("\nDONE ->", OUT_DIR)
